# 第 6 週實作：SVM 與核方法視覺化
# Week 6 Lab: SVM & Kernel Methods Visualization

## 學習目標
- 視覺化線性 SVM 的決策邊界與支持向量
- 觀察正則化參數 C 對間隔和決策邊界的影響
- 比較不同核函數 (Linear, Polynomial, RBF, Sigmoid) 的決策邊界
- 理解 RBF 核中 gamma 參數的作用
- 使用 C-gamma 熱力圖搜索最佳超參數
- 在非線性資料集 (Moon / Circle) 上實作 SVM

In [ ]:
# 環境準備 Environment Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn import svm
from sklearn.svm import SVC, SVR
from sklearn.datasets import make_moons, make_circles, make_classification, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型與繪圖風格
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print('所有套件載入成功！All packages imported successfully!')

---
## 1. 線性 SVM 決策邊界與支持向量視覺化
### Linear SVM: Decision Boundary & Support Vectors

首先建立一個簡單的線性可分資料集，觀察 SVM 如何找到最大間隔的超平面。

In [ ]:
def plot_svm_decision_boundary(clf, X, y, ax=None, title='SVM Decision Boundary',
                                show_margin=True, show_sv=True):
    """繪製 SVM 決策邊界、間隔與支持向量的通用函數"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # 建立網格
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))

    # 決策函數值
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # 繪製決策區域
    ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 50),
                cmap='RdBu', alpha=0.3)

    # 繪製決策邊界 (f(x)=0) 和間隔邊界 (f(x)=+/-1)
    ax.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2, linestyles='-')
    if show_margin:
        ax.contour(xx, yy, Z, levels=[-1, 1], colors='k', linewidths=1, linestyles='--')

    # 繪製資料點
    colors = ['#2563eb', '#dc2626']
    markers = ['o', 's']
    for idx, cls in enumerate(np.unique(y)):
        mask = y == cls
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[idx], marker=markers[idx],
                   edgecolors='k', s=60, label=f'Class {cls}', zorder=3)

    # 標記支持向量
    if show_sv:
        ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1],
                   s=200, facecolors='none', edgecolors='#f59e0b', linewidths=2.5,
                   label=f'Support Vectors (n={len(clf.support_vectors_)})', zorder=4)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.set_xlabel('Feature $x_1$')
    ax.set_ylabel('Feature $x_2$')
    return ax

In [ ]:
# 生成線性可分資料
np.random.seed(42)
X_linear, y_linear = make_blobs(n_samples=100, centers=2, cluster_std=1.2,
                                 random_state=42)

# 訓練線性 SVM
clf_linear = SVC(kernel='linear', C=1.0)
clf_linear.fit(X_linear, y_linear)

# 視覺化
fig, ax = plt.subplots(figsize=(9, 7))
plot_svm_decision_boundary(clf_linear, X_linear, y_linear, ax=ax,
                            title='線性 SVM：決策邊界、間隔與支持向量\nLinear SVM: Decision Boundary, Margin & Support Vectors')

# 標註間隔寬度
w = clf_linear.coef_[0]
margin_width = 2 / np.linalg.norm(w)
ax.text(0.02, 0.02, f'Margin Width = {margin_width:.3f}\n'
        f'Support Vectors: {len(clf_linear.support_vectors_)}\n'
        f'w = [{w[0]:.3f}, {w[1]:.3f}]\n'
        f'b = {clf_linear.intercept_[0]:.3f}',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print(f'\n支持向量的索引: {clf_linear.support_}')
print(f'每類的支持向量數量: {clf_linear.n_support_}')
print(f'訓練準確率: {clf_linear.score(X_linear, y_linear):.4f}')

### 觀察與思考
- 黑色實線是決策邊界 ($f(x) = 0$)
- 虛線是間隔邊界 ($f(x) = \pm 1$)
- 金色圈圈標記的是支持向量 (Support Vectors)
- 只有支持向量決定了邊界的位置

> **試一試：** 修改 `cluster_std` 的值，觀察資料重疊程度對支持向量數量的影響。

---
## 2. C 值調整實驗
### Effect of Regularization Parameter C

C 控制了「寬間隔」vs「少錯誤」之間的取捨。讓我們比較 C = 0.01, 1, 100 三個值。

In [ ]:
# 生成稍有重疊的資料（不完全線性可分）
np.random.seed(0)
X_overlap, y_overlap = make_blobs(n_samples=150, centers=2, cluster_std=2.0,
                                   random_state=0)

# 比較不同 C 值
C_values = [0.01, 1, 100]
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, C in zip(axes, C_values):
    clf = SVC(kernel='linear', C=C)
    clf.fit(X_overlap, y_overlap)

    plot_svm_decision_boundary(clf, X_overlap, y_overlap, ax=ax,
                                title=f'C = {C}')

    w = clf.coef_[0]
    margin_width = 2 / np.linalg.norm(w)
    train_acc = clf.score(X_overlap, y_overlap)
    n_sv = len(clf.support_vectors_)

    ax.text(0.02, 0.02,
            f'Margin = {margin_width:.2f}\nSVs = {n_sv}\nAcc = {train_acc:.3f}',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle('正則化參數 C 的影響 — Effect of C on Linear SVM',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- C=0.01: 寬間隔，允許較多錯誤，支持向量多')
print('- C=1:    平衡間隔與準確度')
print('- C=100:  窄間隔，嚴格分類，支持向量少但模型可能過擬合')

In [ ]:
# 更細緻的 C 值掃描：觀察間隔寬度、支持向量數量與準確率的變化趨勢
C_range = np.logspace(-3, 3, 30)
margins = []
n_svs = []
accuracies = []

for C in C_range:
    clf = SVC(kernel='linear', C=C)
    clf.fit(X_overlap, y_overlap)
    w = clf.coef_[0]
    margins.append(2 / np.linalg.norm(w))
    n_svs.append(len(clf.support_vectors_))
    accuracies.append(clf.score(X_overlap, y_overlap))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].semilogx(C_range, margins, 'o-', color='#2563eb', markersize=4)
axes[0].set_xlabel('C (log scale)')
axes[0].set_ylabel('Margin Width')
axes[0].set_title('C vs 間隔寬度 Margin Width')
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(C_range, n_svs, 's-', color='#dc2626', markersize=4)
axes[1].set_xlabel('C (log scale)')
axes[1].set_ylabel('Number of Support Vectors')
axes[1].set_title('C vs 支持向量數量')
axes[1].grid(True, alpha=0.3)

axes[2].semilogx(C_range, accuracies, '^-', color='#059669', markersize=4)
axes[2].set_xlabel('C (log scale)')
axes[2].set_ylabel('Training Accuracy')
axes[2].set_title('C vs 訓練準確率')
axes[2].set_ylim([min(accuracies) - 0.02, 1.01])
axes[2].grid(True, alpha=0.3)

fig.suptitle('C 值掃描分析 — C Parameter Sweep Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 觀察結論
- C 增大 → 間隔變窄 → 支持向量減少 → 訓練準確率上升
- 但 C 太大可能過擬合！在實際應用中需要用交叉驗證 (Cross-Validation) 選擇最佳 C

---
## 3. 核函數比較視覺化
### Kernel Function Comparison

使用同一個非線性資料集，比較四種核函數的決策邊界。

In [ ]:
# 生成非線性資料 (moon dataset)
np.random.seed(42)
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)

# 特徵縮放
scaler = StandardScaler()
X_moon_scaled = scaler.fit_transform(X_moon)

# 四種核函數
kernels = [
    ('Linear', SVC(kernel='linear', C=1)),
    ('Polynomial (d=3)', SVC(kernel='poly', degree=3, C=1, coef0=1)),
    ('RBF (Gaussian)', SVC(kernel='rbf', C=1, gamma='scale')),
    ('Sigmoid', SVC(kernel='sigmoid', C=1, gamma='scale', coef0=0))
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, (name, clf) in zip(axes, kernels):
    clf.fit(X_moon_scaled, y_moon)
    acc = clf.score(X_moon_scaled, y_moon)
    n_sv = len(clf.support_vectors_)

    plot_svm_decision_boundary(clf, X_moon_scaled, y_moon, ax=ax,
                                title=f'{name}\nAcc={acc:.3f}, SVs={n_sv}',
                                show_margin=False)

fig.suptitle('四種核函數在 Moon 資料集上的決策邊界比較\nKernel Comparison on Moon Dataset',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('比較觀察：')
print('- Linear: 只能畫直線，無法處理彎月形資料')
print('- Polynomial: 可以畫曲線，但形狀受限於多項式次數')
print('- RBF: 最靈活，可以擬合任意形狀的邊界')
print('- Sigmoid: 效果通常不如 RBF，較少使用')

In [ ]:
# 再用 Circle 資料集比較
np.random.seed(42)
X_circle, y_circle = make_circles(n_samples=200, noise=0.1, factor=0.4, random_state=42)
X_circle_scaled = StandardScaler().fit_transform(X_circle)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

kernels_circle = [
    ('Linear', SVC(kernel='linear', C=1)),
    ('Polynomial (d=2)', SVC(kernel='poly', degree=2, C=1, coef0=1)),
    ('RBF (Gaussian)', SVC(kernel='rbf', C=10, gamma='scale')),
    ('Polynomial (d=5)', SVC(kernel='poly', degree=5, C=1, coef0=1))
]

for ax, (name, clf) in zip(axes, kernels_circle):
    clf.fit(X_circle_scaled, y_circle)
    acc = clf.score(X_circle_scaled, y_circle)
    n_sv = len(clf.support_vectors_)

    plot_svm_decision_boundary(clf, X_circle_scaled, y_circle, ax=ax,
                                title=f'{name}\nAcc={acc:.3f}, SVs={n_sv}',
                                show_margin=False)

fig.suptitle('核函數在 Circle 資料集上的比較\nKernel Comparison on Circle Dataset',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('觀察重點：')
print('- Linear 完全無法分離環形資料')
print('- Polynomial (d=2) 因為 x1^2 + x2^2 恰好適合圓形分佈，效果很好')
print('- RBF 也能良好處理環形資料')

---
## 4. RBF 核的 gamma 影響視覺化
### Visualizing the Effect of gamma in RBF Kernel

gamma 控制每個資料點的「影響範圍」。小 gamma = 大範圍（平滑），大 gamma = 小範圍（複雜）。

In [ ]:
# gamma 對 RBF-SVM 決策邊界的影響
gamma_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for ax, gamma in zip(axes, gamma_values):
    clf = SVC(kernel='rbf', C=1, gamma=gamma)
    clf.fit(X_moon_scaled, y_moon)
    acc = clf.score(X_moon_scaled, y_moon)
    n_sv = len(clf.support_vectors_)

    plot_svm_decision_boundary(clf, X_moon_scaled, y_moon, ax=ax,
                                title=f'gamma = {gamma}\nAcc={acc:.3f}, SVs={n_sv}',
                                show_margin=False, show_sv=False)

fig.suptitle('RBF 核的 gamma 參數影響 — Effect of gamma on RBF-SVM',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

print('gamma 影響分析：')
print('- gamma=0.01: 邊界過於平滑，幾乎是直線 (欠擬合 Underfitting)')
print('- gamma=0.1:  邊界開始彎曲，但仍較簡單')
print('- gamma=1:    適度彎曲，可能是較佳選擇')
print('- gamma=10:   邊界開始出現不規則彎曲')
print('- gamma=100:  每個支持向量形成自己的「島」(嚴重過擬合 Overfitting)')

In [ ]:
# RBF 核函數可視化：gamma 如何影響「山丘」的形狀
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = np.linspace(-3, 3, 300)
gammas_demo = [0.1, 1, 10]
colors_demo = ['#2563eb', '#dc2626', '#059669']

for ax, gamma, color in zip(axes, gammas_demo, colors_demo):
    # K(x, 0) = exp(-gamma * x^2)
    K = np.exp(-gamma * x**2)
    ax.plot(x, K, color=color, linewidth=2.5)
    ax.fill_between(x, K, alpha=0.2, color=color)
    ax.set_title(f'RBF Kernel (gamma={gamma})', fontsize=12, fontweight='bold')
    ax.set_xlabel('$\\|x - x_i\\|$')
    ax.set_ylabel('$K(x, x_i)$')
    ax.set_ylim([-0.05, 1.1])
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='gray', linewidth=0.5)

fig.suptitle('RBF 核函數形狀：gamma 越大，影響範圍越小\n'
             'RBF Kernel Shape: Larger gamma = Narrower influence',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. C-gamma 搜尋熱力圖
### C-gamma Grid Search Heatmap

使用 GridSearchCV 搜索最佳的 (C, gamma) 組合，並以熱力圖視覺化。

In [ ]:
# 準備資料（使用 Moon 資料集）
X_train, X_test, y_train, y_test = train_test_split(
    X_moon_scaled, y_moon, test_size=0.3, random_state=42)

# 定義搜索空間（對數尺度）
C_range = np.logspace(-2, 3, 12)       # 0.01 到 1000
gamma_range = np.logspace(-3, 2, 12)   # 0.001 到 100

param_grid = {'C': C_range, 'gamma': gamma_range}

# 網格搜索 + 5-fold 交叉驗證
grid_search = GridSearchCV(
    SVC(kernel='rbf'),
    param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f'最佳參數 Best Parameters: {grid_search.best_params_}')
print(f'最佳交叉驗證準確率 Best CV Accuracy: {grid_search.best_score_:.4f}')
print(f'測試集準確率 Test Accuracy: {grid_search.score(X_test, y_test):.4f}')

In [ ]:
# 繪製 C-gamma 熱力圖
scores = grid_search.cv_results_['mean_test_score'].reshape(len(C_range), len(gamma_range))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左圖：CV 準確率熱力圖
im1 = axes[0].imshow(scores, interpolation='nearest', cmap='YlOrRd', aspect='auto')
axes[0].set_xlabel('gamma (log scale)', fontsize=11)
axes[0].set_ylabel('C (log scale)', fontsize=11)
axes[0].set_xticks(range(len(gamma_range)))
axes[0].set_xticklabels([f'{g:.0e}' for g in gamma_range], rotation=45, fontsize=8)
axes[0].set_yticks(range(len(C_range)))
axes[0].set_yticklabels([f'{c:.0e}' for c in C_range], fontsize=8)
axes[0].set_title('交叉驗證準確率 CV Accuracy Heatmap', fontsize=12, fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='Mean CV Accuracy')

# 標記最佳點
best_C_idx = np.where(C_range == grid_search.best_params_['C'])[0][0]
best_g_idx = np.where(gamma_range == grid_search.best_params_['gamma'])[0][0]
axes[0].scatter(best_g_idx, best_C_idx, marker='*', s=300, c='white', edgecolors='black',
               linewidths=2, zorder=5)
axes[0].annotate('Best', (best_g_idx, best_C_idx), fontsize=10, fontweight='bold',
                color='white', ha='center', va='bottom',
                xytext=(0, 15), textcoords='offset points')

# 右圖：使用 Seaborn 熱力圖（帶數值標注）
scores_df = pd.DataFrame(scores,
                         index=[f'{c:.1e}' for c in C_range],
                         columns=[f'{g:.1e}' for g in gamma_range])
sns.heatmap(scores_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[1],
            annot_kws={'fontsize': 7}, cbar_kws={'label': 'Mean CV Accuracy'})
axes[1].set_xlabel('gamma', fontsize=11)
axes[1].set_ylabel('C', fontsize=11)
axes[1].set_title('C-gamma 搜尋結果 (附數值)\nGrid Search Results (with values)',
                  fontsize=12, fontweight='bold')

fig.suptitle('C-gamma 網格搜索熱力圖 — Grid Search Heatmap',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n熱力圖解讀：')
print('- 顏色越深（紅/橙）= 準確率越高')
print('- 星號標記最佳 (C, gamma) 組合')
print('- 右上角（大C + 大gamma）往往過擬合')
print('- 左下角（小C + 小gamma）往往欠擬合')

In [ ]:
# 使用最佳參數的模型，繪製最終決策邊界
best_clf = grid_search.best_estimator_

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# 訓練集
plot_svm_decision_boundary(best_clf, X_train, y_train, ax=axes[0],
                            title=f'訓練集 Training Set\nC={grid_search.best_params_["C"]:.2f}, '
                                  f'gamma={grid_search.best_params_["gamma"]:.4f}',
                            show_margin=False)

# 測試集
plot_svm_decision_boundary(best_clf, X_test, y_test, ax=axes[1],
                            title=f'測試集 Test Set\nAccuracy = {best_clf.score(X_test, y_test):.3f}',
                            show_margin=False)

fig.suptitle('最佳 RBF-SVM 模型的決策邊界\nBest RBF-SVM Decision Boundary',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. 非線性資料集分類
### Non-linear Dataset Classification (Moon & Circle)

讓我們在 Moon 和 Circle 兩個經典非線性資料集上完整演示 SVM 的威力。

In [ ]:
# 建立 Moon 和 Circle 資料集
datasets = [
    ('Moon (noise=0.2)', make_moons(n_samples=300, noise=0.2, random_state=42)),
    ('Moon (noise=0.35)', make_moons(n_samples=300, noise=0.35, random_state=42)),
    ('Circle (noise=0.1)', make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42)),
    ('Circle (noise=0.2)', make_circles(n_samples=300, noise=0.2, factor=0.4, random_state=42)),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))

for col, (name, (X_data, y_data)) in enumerate(datasets):
    X_scaled = StandardScaler().fit_transform(X_data)

    # 上排：Linear SVM
    clf_lin = SVC(kernel='linear', C=1)
    clf_lin.fit(X_scaled, y_data)
    plot_svm_decision_boundary(clf_lin, X_scaled, y_data, ax=axes[0, col],
                                title=f'{name}\nLinear (Acc={clf_lin.score(X_scaled, y_data):.2f})',
                                show_margin=False, show_sv=False)

    # 下排：RBF SVM
    clf_rbf = SVC(kernel='rbf', C=10, gamma='scale')
    clf_rbf.fit(X_scaled, y_data)
    plot_svm_decision_boundary(clf_rbf, X_scaled, y_data, ax=axes[1, col],
                                title=f'{name}\nRBF (Acc={clf_rbf.score(X_scaled, y_data):.2f})',
                                show_margin=False, show_sv=False)

axes[0, 0].set_ylabel('Linear SVM', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('RBF SVM', fontsize=12, fontweight='bold')

fig.suptitle('線性 vs RBF SVM 在非線性資料集上的表現\n'
             'Linear vs RBF SVM on Non-linear Datasets',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('結論：')
print('- Linear SVM 無法處理非線性資料')
print('- RBF SVM 透過核技巧可以優雅地處理 Moon 和 Circle 資料集')
print('- 噪音增大時，兩種方法的準確率都下降，但 RBF 仍然表現較好')

---
## 7. 進階探索：核技巧的幾何直覺
### Advanced: Geometric Intuition of Kernel Trick

以 Circle 資料集為例，展示如何透過映射到高維空間使資料線性可分。

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# 使用 Circle 資料集
np.random.seed(42)
X_circ, y_circ = make_circles(n_samples=200, noise=0.05, factor=0.4, random_state=42)

fig = plt.figure(figsize=(16, 5.5))

# 左圖：原始 2D 空間（無法線性分離）
ax1 = fig.add_subplot(131)
for cls, color, marker in [(0, '#2563eb', 'o'), (1, '#dc2626', 's')]:
    mask = y_circ == cls
    ax1.scatter(X_circ[mask, 0], X_circ[mask, 1], c=color, marker=marker,
               edgecolors='k', s=40, label=f'Class {cls}')
ax1.set_title('原始 2D 空間\n(無法線性分離)', fontsize=12, fontweight='bold')
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# 中圖：映射到 3D 空間 (x1, x2, x1^2 + x2^2)
ax2 = fig.add_subplot(132, projection='3d')
z = X_circ[:, 0]**2 + X_circ[:, 1]**2  # 新特徵
for cls, color, marker in [(0, '#2563eb', 'o'), (1, '#dc2626', 's')]:
    mask = y_circ == cls
    ax2.scatter(X_circ[mask, 0], X_circ[mask, 1], z[mask],
               c=color, marker=marker, s=30, alpha=0.7, edgecolors='k', linewidths=0.5)
ax2.set_title('映射到 3D 空間\n$\phi(x_1, x_2) = (x_1, x_2, x_1^2+x_2^2)$',
             fontsize=11, fontweight='bold')
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_zlabel('$x_1^2 + x_2^2$')
ax2.view_init(elev=20, azim=30)

# 在 3D 空間畫分離平面
xx_plane = np.linspace(-1.5, 1.5, 10)
yy_plane = np.linspace(-1.5, 1.5, 10)
xx_plane, yy_plane = np.meshgrid(xx_plane, yy_plane)
threshold = 0.5 * (z[y_circ == 0].mean() + z[y_circ == 1].mean())
zz_plane = np.full_like(xx_plane, threshold)
ax2.plot_surface(xx_plane, yy_plane, zz_plane, alpha=0.2, color='green')

# 右圖：RBF SVM 在原始空間的決策邊界
ax3 = fig.add_subplot(133)
clf_demo = SVC(kernel='rbf', C=10, gamma=2)
clf_demo.fit(X_circ, y_circ)
plot_svm_decision_boundary(clf_demo, X_circ, y_circ, ax=ax3,
                            title=f'RBF SVM 在原始空間的邊界\nAcc={clf_demo.score(X_circ, y_circ):.3f}',
                            show_margin=False)

fig.suptitle('核技巧的幾何直覺 — Geometric Intuition of Kernel Trick',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print('直覺理解：')
print('1. 左圖：在 2D 中，圓內(紅)和圓外(藍)無法用直線分開')
print('2. 中圖：加入 x1^2 + x2^2 作為第三維，資料在 3D 中被「拉開」')
print('   → 綠色平面可以分開兩類！')
print('3. 右圖：RBF 核自動完成了這個映射，在原始空間畫出圓形邊界')

---
## 8. SVM vs Logistic Regression 比較
### SVM vs Logistic Regression Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression

# 使用 Moon 資料集比較
X_comp, y_comp = make_moons(n_samples=300, noise=0.25, random_state=42)
X_comp_scaled = StandardScaler().fit_transform(X_comp)
X_tr, X_te, y_tr, y_te = train_test_split(X_comp_scaled, y_comp, test_size=0.3, random_state=42)

models = [
    ('Logistic Regression', LogisticRegression(C=1)),
    ('Linear SVM', SVC(kernel='linear', C=1)),
    ('RBF SVM', SVC(kernel='rbf', C=10, gamma='scale')),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

for ax, (name, model) in zip(axes, models):
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    test_acc = model.score(X_te, y_te)

    # 繪製決策區域
    x_min, x_max = X_comp_scaled[:, 0].min() - 0.5, X_comp_scaled[:, 0].max() + 0.5
    y_min, y_max = X_comp_scaled[:, 1].min() - 0.5, X_comp_scaled[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))

    if hasattr(model, 'decision_function'):
        Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    else:
        Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1] - 0.5
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0], colors='k', linewidths=2)

    for cls, color, marker in [(0, '#2563eb', 'o'), (1, '#dc2626', 's')]:
        mask = y_te == cls
        ax.scatter(X_te[mask, 0], X_te[mask, 1], c=color, marker=marker,
                   edgecolors='k', s=50, zorder=3)

    ax.set_title(f'{name}\nTrain: {train_acc:.3f}, Test: {test_acc:.3f}',
                fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature $x_1$')
    ax.set_ylabel('Feature $x_2$')

fig.suptitle('SVM vs Logistic Regression 在 Moon 資料集上的比較\n'
             'SVM vs Logistic Regression Comparison on Moon Dataset',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('比較結果：')
print('- Logistic Regression: 線性邊界，無法捕捉非線性模式')
print('- Linear SVM: 同樣是線性邊界，表現類似 LR')
print('- RBF SVM: 非線性邊界，能良好地分離 Moon 資料')

---
## 9. 練習題 Exercises

### 練習 1：C 值的影響 (Effect of C)
使用 `make_circles` 產生環形資料集 (`noise=0.15, factor=0.5`)，
分別使用 RBF SVM 在 `C = [0.1, 1, 10, 100, 1000]` 五個值下訓練，
並繪製 5 個子圖比較決策邊界。

**提示：** 記得先用 `StandardScaler` 縮放特徵。

In [ ]:
# 練習 1：在這裡撰寫你的程式碼
# TODO: 產生資料、標準化、訓練 5 個不同 C 的 RBF SVM、繪圖比較


### 練習 2：完整的模型選擇流程 (Full Model Selection Pipeline)
使用 Iris 資料集（僅取前 2 個特徵以便視覺化）：
1. 將資料分為訓練集 (70%) 和測試集 (30%)
2. 使用 `GridSearchCV` 搜索最佳的 `kernel`（'linear', 'rbf', 'poly'）、`C`（0.1 到 100）和 `gamma`（0.01 到 10）
3. 輸出最佳參數、交叉驗證準確率、測試集準確率
4. 繪製最佳模型的決策邊界

**提示：** `GridSearchCV(SVC(), param_grid, cv=5)`

In [ ]:
# 練習 2：在這裡撰寫你的程式碼
# TODO: 載入 Iris 資料、分割、GridSearchCV、繪圖
from sklearn.datasets import load_iris


### 練習 3：SVM vs Logistic Regression 定量比較 (Quantitative Comparison)
使用 `make_moons(noise=0.3)` 產生資料，比較以下三種模型的 5-fold 交叉驗證準確率：
1. Logistic Regression
2. Linear SVM
3. RBF SVM (使用你認為合適的 C 和 gamma)

使用 `cross_val_score` 計算，並用箱型圖 (Box Plot) 視覺化三者的分佈。

**提示：** `scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')`

In [ ]:
# 練習 3：在這裡撰寫你的程式碼
# TODO: 產生資料、三種模型的 cross_val_score、Box Plot 比較


### 挑戰題 Challenge：自訂資料集與完整分析

1. 使用 `np.random.seed(你的學號後4碼)` 產生一個自訂資料集（可以是 moon、circle 或自己設計的分佈）
2. 嘗試至少 3 種不同的核函數
3. 使用 GridSearchCV 找到最佳超參數
4. 繪製完整的分析圖（決策邊界 + C-gamma 熱力圖 + 混淆矩陣）
5. 寫一段文字總結你的發現（至少 100 字）

In [ ]:
# 挑戰題：在這裡撰寫你的程式碼


---
## 本週重點回顧 Week 6 Summary

1. **SVM 核心思想**：找到最大間隔 (Maximum Margin) 的超平面
2. **支持向量**：只有間隔邊界上的少數點決定模型 → 稀疏性
3. **參數 C**：控制「寬間隔」vs「少錯誤」的取捨
   - C 大 → 嚴格分類（可能過擬合）
   - C 小 → 寬容（可能欠擬合）
4. **核技巧 (Kernel Trick)**：隱式映射到高維空間，讓非線性問題可以線性分離
5. **RBF 核的 gamma**：控制決策邊界的複雜度
   - gamma 大 → 複雜邊界（可能過擬合）
   - gamma 小 → 平滑邊界（可能欠擬合）
6. **C 和 gamma 需搭配調校**：使用 GridSearchCV + 交叉驗證
7. **特徵縮放**是 SVM 的必要前處理步驟

### 下週預告 Next Week Preview
第 7 週：**樹模型與集成方法** (Decision Tree, Random Forest, GBDT) — 視覺化樹的生長、分裂準則與集成效果！